# Recovery time: Linear Regression

Predicts total recovery time in weeks from age, pain severity and exercise level.

Uses the simulated patients saved by `random_forest/recovery_rf.ipynb` (`data/historical_recovery.csv`) and the same train/test split, so it can be compared fairly with the random forest. Linear Regression doesn't need scaling.

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
sim = pd.read_csv(Path("..") / "data" / "historical_recovery.csv")
print(len(sim))
sim.head()

1000


,Age,pain_severity,exercise_level,recovery_weeks
0,22.0,3.0,0,5.5
1,20.0,4.4,0,5.8
2,23.0,2.7,1,5.7
3,20.0,3.5,3,3.7
4,21.0,3.0,1,3.0


In [3]:
X = sim[['Age', 'pain_severity', 'exercise_level']]
y = sim['recovery_weeks']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_predictions = lr.predict(X_test)
print(f"R²: {r2_score(y_test, lr_predictions):.3f}")
print(f"RMSE: {mean_squared_error(y_test, lr_predictions) ** 0.5:.3f} weeks")

R²: 0.463
RMSE: 1.615 weeks


In [5]:
# Weeks added (or taken away) per unit of each feature
print(f"Intercept: {lr.intercept_:.2f}")
for feature, coef in zip(X.columns, lr.coef_):
    print(f"{feature}: {coef:+.3f} weeks")

Intercept: 2.42
Age: +0.081 weeks
pain_severity: +0.614 weeks
exercise_level: -0.839 weeks


## Comparing with the random forest

Same test set for both models.

In [6]:
rf = joblib.load(Path("..") / "random_forest" / "recovery_rf.joblib")
rf_predictions = rf.predict(X_test)

print(f"Linear Regression  R²: {r2_score(y_test, lr_predictions):.3f}  RMSE: {mean_squared_error(y_test, lr_predictions) ** 0.5:.3f} weeks")
print(f"Random Forest      R²: {r2_score(y_test, rf_predictions):.3f}  RMSE: {mean_squared_error(y_test, rf_predictions) ** 0.5:.3f} weeks")

Linear Regression  R²: 0.463  RMSE: 1.615 weeks
Random Forest      R²: 0.422  RMSE: 1.675 weeks


Cross-validation scores both models on 5 different train/test splits, to check the result isn't luck from one split.

In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for name, model in [("Linear Regression", LinearRegression()),
                    ("Random Forest", RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42))]:
    rmse = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
    print(f"{name}: RMSE {rmse.mean():.3f} weeks (folds {rmse.round(2)}), R² {r2.mean():.3f}")

Linear Regression: RMSE 1.479 weeks (folds [1.61 1.42 1.51 1.52 1.33]), R² 0.499
Random Forest: RMSE 1.552 weeks (folds [1.68 1.52 1.59 1.58 1.39]), R² 0.448


In [8]:
joblib.dump(lr, "recovery_lr.joblib")

['recovery_lr.joblib']